# Phase 6.1: The Robust Clinical Oracle (Improved Classifier)

This notebook trains a **robust 1D ResNet** on the clean biological ground truth for 100 epochs using Cosine Annealing. 
We need the Oracle to actually know how to read an ECG (targeting a Macro F1 > 0.50) before we can trust it to evaluate the Diffusion Model.

**Once trained, this model will be saved as `best_clinical_oracle.pth` so we can use it in Step 2 (Task-Aware Diffusion).**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import ast
import requests
import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# --- 1. Load Data and Labels (Same as before) ---
print("Downloading PTB-XL labels from PhysioNet...")
ptbxl_url = "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv"
scp_url = "https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv"
ptbxl_df = pd.read_csv(io.StringIO(requests.get(ptbxl_url).text))
scp_df = pd.read_csv(io.StringIO(requests.get(scp_url).text), index_col=0)

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in scp_df.index:
            cls = scp_df.loc[key].diagnostic_class
            if str(cls) != 'nan':
                tmp.append(cls)
    return list(set(tmp))

ptbxl_df['scp_codes'] = ptbxl_df['scp_codes'].apply(lambda x: ast.literal_eval(x))
ptbxl_df['diagnostic_superclass'] = ptbxl_df['scp_codes'].apply(aggregate_diagnostic)

metadata = pd.read_csv('/content/metadata.csv')
clean_all = np.load('/content/clean_samples.npy')
noisy_all = np.load('/content/noisy_samples.npy')

metadata['ecg_id_int'] = metadata['ecg_id'].astype(int)
merged = metadata.merge(ptbxl_df[['ecg_id', 'diagnostic_superclass']], left_on='ecg_id_int', right_on='ecg_id', how='left')
valid_indices = merged['diagnostic_superclass'].apply(lambda x: isinstance(x, list) and len(x) > 0)

clean_filtered = clean_all[valid_indices]
noisy_filtered = noisy_all[valid_indices]
labels_raw = merged['diagnostic_superclass'][valid_indices].tolist()

superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
y = np.zeros((len(labels_raw), 5))
for i, labels in enumerate(labels_raw):
    for j, sc in enumerate(superclasses):
        if sc in labels:
            y[i, j] = 1

if len(clean_filtered.shape) == 2:
    clean_filtered = np.expand_dims(clean_filtered, axis=1)
    noisy_filtered = np.expand_dims(noisy_filtered, axis=1)

print(f"Data ready: {len(y)} labeled samples.")

In [ ]:
# --- 2. Build the ResNet Oracle ---
class ResNet1DBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.downsample = nn.Sequential(nn.Conv1d(in_channels, out_channels, 1, stride), nn.BatchNorm1d(out_channels)) if stride != 1 or in_channels != out_channels else nn.Identity()
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.downsample(x)
        return self.relu(out)

class ResNet1D(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        self.layer1 = nn.Sequential(nn.Conv1d(1, 32, 15, stride=2, padding=7), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(3, 2, 1))
        self.layer2 = ResNet1DBlock(32, 64, stride=2)
        self.layer3 = ResNet1DBlock(64, 128, stride=2)
        self.layer4 = ResNet1DBlock(128, 256, stride=2)
        self.layer5 = ResNet1DBlock(256, 512, stride=2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, num_classes)
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

In [ ]:
# --- 3. Train the Oracle Properly ---
X_train, X_test, y_train, y_test = train_test_split(clean_filtered, y, test_size=0.2, random_state=42)
_, noisy_test, _, _ = train_test_split(noisy_filtered, y, test_size=0.2, random_state=42)

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

classifier = ResNet1D(num_classes=5).to(device)

# Calculate positive weights for Imbalanced dataset
pos_weights = (len(y_train) - y_train.sum(axis=0)) / (y_train.sum(axis=0) + 1e-5)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weights, dtype=torch.float32).to(device))

optimizer = optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 100
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

print("Training ROBUST Classifier on CLEAN (Ground Truth) Data...")
best_loss = float('inf')
for epoch in range(1, epochs + 1):
    classifier.train()
    total_loss = 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(classifier(X_b), y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    avg_loss = total_loss/len(train_loader)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(classifier.state_dict(), 'best_clinical_oracle.pth')
        
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.4f}")

print("Saved best_clinical_oracle.pth")

In [ ]:
# --- 4. Evaluate the New Oracle ---
classifier.load_state_dict(torch.load('best_clinical_oracle.pth'))
classifier.eval()

def evaluate(data, labels):
    data_tensor = torch.tensor(data, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = classifier(data_tensor).cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs > 0.5).astype(int)
    return f1_score(labels, preds, average='macro', zero_division=0), roc_auc_score(labels, probs, average='macro')

clean_f1, clean_auroc = evaluate(X_test, y_test)
noisy_f1, noisy_auroc = evaluate(noisy_test, y_test)

print(f"Oracle (Clean) Macro F1: {clean_f1:.4f} | AUROC: {clean_auroc:.4f}")
print(f"Raw Noisy      Macro F1: {noisy_f1:.4f} | AUROC: {noisy_auroc:.4f}")
print("\nIf the Oracle F1 is > 0.50, we are ready for Step 2: Task-Aware Diffusion!")